## Transformaciones: Funciones

Limpieza, claves de identidad y MinHash/LSH

### Imports

In [ ]:
import requests

### Limpieza

- `quitar_duplicados`: 1 fila por `ocid`, la más reciente

- `convertir_a_soles`: tipo de cambio mensual del BCRP

- `limpiar_montos`: monto ≤ 0 → nulo (no es un monto real)

In [ ]:
def normalizar_texto(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.regexp_replace(F.upper(F.trim(F.col(c))), r"\s+", " "))
    return df

def quitar_duplicados(df):
    w = Window.partitionBy("ocid").orderBy(F.col("fecha").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

def limpiar_montos(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.when(F.col(c) > 0, F.col(c)))
    return df

In [ ]:
SERIES_TC = {"USD": "PN01234PM", "EUR": "PN01235PM", "BRL": "PN01237PM", "GBP": "PN01238PM", "CAD": "PN01244PM"}
MESES = {"Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4, "May": 5, "Jun": 6, "Jul": 7, "Ago": 8, "Sep": 9, "Set": 9,
         "Oct": 10, "Nov": 11, "Dic": 12}

def descargar_tipo_cambio(ruta, anios):
    filas = []
    for moneda, serie in SERIES_TC.items():
        url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{serie}/json/{min(anios)}-1/{max(anios)}-12"
        for periodo in requests.get(url, timeout=30).json()["periods"]:
            mes, anio = periodo["name"].split(".")
            filas.append((moneda, int(anio), MESES[mes], float(periodo["values"][0])))
    pd.DataFrame(filas, columns=["moneda", "anio", "_mes", "tc"]).to_csv(ruta, index=False)

def convertir_a_soles(df, tc, col_monto, col_moneda, col_fecha):
    tc_ = tc.withColumnRenamed("moneda", "_mon").withColumnRenamed("anio", "_anio")
    return (df.withColumn("_mon", F.coalesce(F.col(col_moneda), F.lit("PEN")))
              .withColumn("_anio", F.year(col_fecha)).withColumn("_mes", F.month(col_fecha))
              .join(F.broadcast(tc_), ["_mon", "_anio", "_mes"], "left")
              .withColumn(col_monto, F.when(F.col("_mon") == "PEN", F.col(col_monto)).otherwise(F.col(col_monto) * F.col("tc")))
              .drop("_mon", "_anio", "_mes", "tc"))

### Claves de identidad

| Prefijo | Origen | Ejemplo crudo |
|---|---|---|
| `ENT:` | Código CONSUCODE de la entidad | `PE-CONSUCODE-200489` |
| `RUC:` | RUC de persona jurídica (11 dígitos) | `PE-RUC-20546736718` |
| `PN:` | Persona natural: DNI, o RUC 10 (= `10` + DNI + dígito) | `PE-RUC-10xxxxxxxxx` |
| `CONS:` | Código de consorcio del SEACE (6-7 dígitos, no es RUC) | `PE-RUC-1503248` |
| `EXT:` | Proveedor extranjero sin RUC (`L` + 10 dígitos) | `PE-RUC-L0556212336` |
| `NOM:` | Sin identificador: hash del nombre normalizado | — |

In [ ]:
def clave_entidad(c_id, c_nombre):
    cod = F.regexp_extract(F.coalesce(F.col(c_id), F.lit("")), r"PE-CONSUCODE-(\d+)$", 1)
    return F.when(cod != "", F.concat(F.lit("ENT:"), cod)).otherwise(F.concat(F.lit("NOM:"), F.substring(F.sha1(c_nombre), 1, 12)))

def clave_id(c):
    s = F.coalesce(F.col(c).cast("string"), F.lit(""))
    cons = F.regexp_extract(s, r"^PE-RUC-(\d{6,7})$", 1)
    ext = F.regexp_extract(s, r"^PE-RUC-(L\d{10})$", 1)
    d11 = F.regexp_extract(s, r"(\d{11})", 1)
    d8 = F.regexp_extract(s, r"(?<!\d)(\d{8})(?!\d)", 1)
    return (F.when(cons != "", F.concat(F.lit("CONS:"), cons))
             .when(ext != "", F.concat(F.lit("EXT:"), ext))
             .when(d11.startswith("10"), F.concat(F.lit("PN:"), F.substring(d11, 3, 8)))
             .when(d11 != "", F.concat(F.lit("RUC:"), d11))
             .when(d8 != "", F.concat(F.lit("PN:"), d8)))

### Nombre normalizado

- Mayúsculas, sin tildes ni puntuación y **sin forma societaria** (`S.A.C.` = `SOCIEDAD ANONIMA CERRADA` = `SAC`)

In [ ]:
CON_TILDE, SIN_TILDE = "ÁÉÍÓÚÜÑ", "AEIOUUN"
FORMAS = (r"\b(SOCIEDAD ANONIMA CERRADA|SOCIEDAD ANONIMA ABIERTA|SOCIEDAD ANONIMA|"
          r"SOCIEDAD COMERCIAL DE RESPONSABILIDAD LIMITADA|EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA|"
          r"S A C|S A A|S R L|E I R L|SAC|SAA|SRL|EIRL|LTDA|S A)\b")

def normalizar_nombre(c):
    x = F.translate(F.upper(F.trim(F.col(c))), CON_TILDE, SIN_TILDE)
    x = F.regexp_replace(x, r"[^A-Z0-9 ]", " ")
    x = F.regexp_replace(F.regexp_replace(x, r"\s+", " "), FORMAS, " ")
    return F.coalesce(F.trim(F.regexp_replace(x, r"\s+", " ")), F.lit(""))

### MinHash + LSH

- Shingles de **3 caracteres**: los nombres son cortos y k = 5 (el del P1) deja muy pocos shingles

- Firma de `K` = 30 hashes, en `B` = 6 bandas de `R` = 5 filas

- Umbral ≈ (1/B)^(1/R) ≈ 0.70; P(candidato | s = 0.85) ≈ 0.97 y P(candidato | s = 0.5) ≈ 0.17

In [ ]:
K, B = 30, 6
R = K // B

def con_shingles(df, k=3):
    return df.withColumn("sh", F.when(F.length("nombre_norm") >= k, F.array_distinct(
        F.expr(f"transform(sequence(1, length(nombre_norm) - {k} + 1), i -> substring(nombre_norm, i, {k}))")))
        .otherwise(F.array("nombre_norm")))

def minhash(i):
    return F.array_min(F.transform("sh", lambda s: F.xxhash64(s, F.lit(i))))

def bandas_lsh(sh):
    llaves = [F.xxhash64(*[F.col("firma")[b * R + j] for j in range(R)]) for b in range(B)]
    return (sh.withColumn("firma", F.array(*[minhash(i) for i in range(K)]))
              .select("nombre_norm", F.posexplode(F.array(*llaves)).alias("banda", "bucket")))

def jaccard(a, b):
    return F.size(F.array_intersect(a, b)) / F.size(F.array_union(a, b))

### Materializar

- Escribe y relee un resultado intermedio

In [ ]:
tmp_dir = proyecto / "data" / "tmp"

def materializar(df, nombre):
    df.write.mode("overwrite").parquet(str(tmp_dir / nombre))
    return SparkSession.getActiveSession().read.parquet(str(tmp_dir / nombre))